# 交互式图表和异步编程
Matplotlib 通过将图表嵌入到 GUI 窗口中，支持丰富的交互式图表。在坐标轴（Axes）上进行平移和缩放以查看数据的基本交互功能已经“内置”于 Matplotlib 中。这得益于全面的鼠标和键盘事件处理系统，你可以使用它来构建复杂的交互式图表。

本指南旨在介绍 Matplotlib 与 GUI 事件循环集成的底层细节。对于更实用的 Matplotlib 事件 API 的介绍，请参阅[事件处理系统](event-handling)、[交互式教程](https://github.com/matplotlib/interactive_tutorial)和[使用 Matplotlib 的交互式应用程序](http://www.amazon.com/Interactive-Applications-using-Matplotlib-Benjamin/dp/1783988843)。

## 事件循环
本质上，所有用户交互（和网络）都是通过无限循环来实现的，该循环等待来自用户（通过操作系统）的事件，然后对其做出响应。例如，简单的读取-求值-输出循环（REPL）是：

```python
exec_count = 0
while True:
    inp = input(f"[{exec_count}] > ")        # Read
    ret = eval(inp)                          # Evaluate
    print(ret)                               # Print
    exec_count += 1                          # Loop
```

这缺少了许多细节（例如，它在遇到第一个异常时就退出！），但它代表了所有终端、图形用户界面和服务器背后的事件循环的基础。通常，读取步骤是在等待某种输入/输出（I/O）——无论是用户输入还是网络通信——而评估和打印步骤则负责解释输入并据此采取行动。

实际上，我们与框架进行交互，该框架提供了一种机制来注册回调函数，以响应特定事件，而不是直接实现输入/输出循环。例如，“当用户点击这个按钮时，请运行这个函数”或“当用户按下‘`z`’键时，请运行另一个函数”。这允许用户编写反应式、事件驱动的程序，而无需深入了解输入/输出的细节。核心事件循环有时被称为“主循环”，通常根据库的不同，通过名为 `_exec`、`run` 或 `start` 的方法启动。

所有图形用户界面框架（`Qt`、`Wx`、`Gtk`、`tk`、macOS 或 Web）都有某种方法捕获用户交互并将它们传回应用程序（例如 `Qt` 中的 `Signal` / `Slot`），但具体细节取决于工具包。Matplotlib 为我们支持的每个  GUI 工具包提供了[后端](https://matplotlib.org/stable/users/explain/figure/backends.html#what-is-a-backend)，使用工具包 API 将工具包的用户界面事件桥接到 Matplotlib 的事件处理系统。然后，您可以使用 {meth}`FigureCanvasBase.mpl_connect` 将您的函数连接到 Matplotlib 的事件处理系统。这使您能够直接与数据交互，并编写不依赖于特定 GUI 工具包的用户界面。